In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# SigLIP: TRAINING + EVALUATION

!pip install -q transformers datasets torch scikit-learn pillow

In [3]:
# ---- IMPORTS ----
import torch, numpy as np
from transformers import SiglipProcessor, SiglipModel
from sklearn.metrics import accuracy_score, f1_score
from torch.nn.functional import normalize

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# ---- HF LOGIN ----
from huggingface_hub import login
login(token="your_token")


In [7]:
# ---- LOAD DATASET ----
from datasets import load_dataset
ds = load_dataset("Navarasa-9/navarasa_9")

labels = sorted(set(ds["train"]["navarasa"]))
label2id = {l: i for i, l in enumerate(labels)}

ds = ds.map(lambda x: {"label": label2id[x["navarasa"]]})

README.md:   0%|          | 0.00/2.04k [00:00<?, ?B/s]

data/train-00000-of-00007.parquet:   0%|          | 0.00/480M [00:00<?, ?B/s]

data/train-00001-of-00007.parquet:   0%|          | 0.00/503M [00:00<?, ?B/s]

data/train-00002-of-00007.parquet:   0%|          | 0.00/500M [00:00<?, ?B/s]

data/train-00003-of-00007.parquet:   0%|          | 0.00/502M [00:00<?, ?B/s]

...

data/test-00000-of-00001.parquet:   0%|          | 0.00/387M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12532 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1393 [00:00<?, ? examples/s]

Map:   0%|          | 0/12532 [00:00<?, ? examples/s]

Map:   0%|          | 0/1393 [00:00<?, ? examples/s]

In [8]:
# ---- METRICS ----
def recall_at_k(sim, k):
    return sum(i in sim[i].topk(k).indices for i in range(sim.size(0))) / sim.size(0)

def retrieval_metrics(img_emb, txt_emb):
    sim = img_emb @ txt_emb.T
    return {
        "R@1": recall_at_k(sim, 1),
        "R@5": recall_at_k(sim, 5),
        "R@10": recall_at_k(sim, 10),
    }

In [9]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    images = [x["image"] for x in batch]
    texts  = [x["text"] for x in batch]
    labels = [x["label"] for x in batch]

    return images, texts, torch.tensor(labels)

In [10]:
train_loader = DataLoader(
    ds["train"],
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn
)

In [11]:
# ---- MODEL ----
processor = SiglipProcessor.from_pretrained("google/siglip-base-patch16-224")
model = SiglipModel.from_pretrained("google/siglip-base-patch16-224").to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/711 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/813M [00:00<?, ?B/s]

In [12]:
!pip install -q tqdm

In [13]:
from tqdm import tqdm

In [14]:
num_epochs = 10

model.train()
for epoch in range(num_epochs):
    epoch_loss = 0.0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{num_epochs}",
        leave=True
    )

    for step, (images, texts, _) in enumerate(progress_bar):
        inputs = processor(
            images=images,
            text=texts,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        outputs = model(**inputs, return_loss=True)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        epoch_loss += loss.item()

        # 🔹 update tqdm bar
        progress_bar.set_postfix({
            "step": step,
            "loss": f"{loss.item():.4f}"
        })

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Avg Loss: {epoch_loss / len(train_loader):.4f}"
    )

Epoch 1/10: 100%|██████████| 784/784 [09:00<00:00,  1.45it/s, step=783, loss=0.3954]


Epoch [1/10] Avg Loss: 1.6471


Epoch 2/10: 100%|██████████| 784/784 [08:56<00:00,  1.46it/s, step=783, loss=0.8173]


Epoch [2/10] Avg Loss: 0.8502


Epoch 3/10: 100%|██████████| 784/784 [08:56<00:00,  1.46it/s, step=783, loss=0.2912]


...

Epoch [8/10] Avg Loss: 0.4918


Epoch 9/10: 100%|██████████| 784/784 [08:56<00:00,  1.46it/s, step=783, loss=0.0495]


Epoch [9/10] Avg Loss: 0.4910


Epoch 10/10: 100%|██████████| 784/784 [08:56<00:00,  1.46it/s, step=783, loss=0.1648]

Epoch [10/10] Avg Loss: 0.4657


In [15]:
siglip_save_dir = "/content/drive/MyDrive/models/siglip-navarasa"

model.save_pretrained(siglip_save_dir)
processor.save_pretrained(siglip_save_dir)

print("SigLIP model saved to Google Drive at:", siglip_save_dir)

SigLIP model saved to Google Drive at: /content/drive/MyDrive/models/siglip-navarasa


In [16]:
!zip -r /content/clip-navarasa.zip /content/drive/MyDrive/models/siglip-navarasa

  adding: content/drive/MyDrive/models/siglip-navarasa/ (stored 0%)
  adding: content/drive/MyDrive/models/siglip-navarasa/config.json (deflated 64%)
  adding: content/drive/MyDrive/models/siglip-navarasa/model.safetensors (deflated 7%)
  adding: content/drive/MyDrive/models/siglip-navarasa/preprocessor_config.json (deflated 50%)
  adding: content/drive/MyDrive/models/siglip-navarasa/tokenizer_config.json (deflated 63%)
  adding: content/drive/MyDrive/models/siglip-navarasa/special_tokens_map.json (deflated 72%)
  adding: content/drive/MyDrive/models/siglip-navarasa/spiece.model (deflated 49%)


In [18]:
from google.colab import files

files.download("/content/siglip-navarasa.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
# ---- EVALUATE ----
model.eval()
img_emb, txt_emb, y_true, y_pred = [], [], [], []

with torch.no_grad():
    for s in ds["test"]:
        img = processor(images=s["image"], return_tensors="pt").to(device)
        txt = processor(text=s["text"], return_tensors="pt").to(device)

        i = normalize(model.get_image_features(**img), dim=-1)
        t = normalize(model.get_text_features(**txt), dim=-1)

        img_emb.append(i)
        txt_emb.append(t)

        y_pred.append((i @ t.T).argmax().item())
        y_true.append(s["label"])

img_emb = torch.cat(img_emb)
txt_emb = torch.cat(txt_emb)

print("\n[SigLIP Evaluation]")
print("Retrieval:", retrieval_metrics(img_emb, txt_emb))
print("Accuracy:", accuracy_score(y_true, y_pred))
print("F1 Macro:", f1_score(y_true, y_pred, average="macro"))
print("F1 Weighted:", f1_score(y_true, y_pred, average="weighted"))



[SigLIP Evaluation]
Retrieval: {'R@1': 0.18951902368987797, 'R@5': 0.5843503230437904, 'R@10': 0.7997128499641063}
Accuracy: 0.05671213208901651
F1 Macro: 0.013417119565217392
F1 Weighted: 0.006087307656293893


In [5]:
import torch
from transformers import SiglipModel, SiglipProcessor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_PATH = "/content/drive/MyDrive/models/siglip-navarasa"

model = SiglipModel.from_pretrained(MODEL_PATH).to(DEVICE)
processor = SiglipProcessor.from_pretrained(MODEL_PATH)

model.eval()

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


SiglipModel(
  (text_model): SiglipTextTransformer(
    (embeddings): SiglipTextEmbeddings(
      (token_embedding): Embedding(32000, 768)
      (position_embedding): Embedding(64, 768)
    )
    (encoder): SiglipEncoder(
      (layers): ModuleList(
        (0-11): 12 x SiglipEncoderLayer(
          (layer_norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (self_attn): SiglipAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): SiglipMLP(
            (activation_fn): GELUTanh()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features

In [6]:
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader

class NavarasaDataset(Dataset):
    def __init__(self, csv_path):
        self.data = pd.read_csv(csv_path)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        text = row["text_prompt"]
        label = row["label"]
        return image, text, label


In [7]:
from tqdm import tqdm
import numpy as np

def encode_dataset(dataloader):
    image_embeds = []
    text_embeds = []
    labels = []

    with torch.no_grad():
        for images, texts, lbls in tqdm(dataloader):
            inputs = processor(
                images=list(images),
                text=list(texts),
                return_tensors="pt",
                padding=True
            ).to(DEVICE)

            outputs = model(**inputs)

            image_embeds.append(outputs.image_embeds.cpu())
            text_embeds.append(outputs.text_embeds.cpu())
            labels.extend(lbls)

    image_embeds = torch.cat(image_embeds)
    text_embeds = torch.cat(text_embeds)

    return image_embeds, text_embeds, labels


In [8]:
import torch.nn.functional as F

def retrieval_metrics(image_embeds, text_embeds):
    image_embeds = F.normalize(image_embeds, dim=1)
    text_embeds = F.normalize(text_embeds, dim=1)

    similarity = image_embeds @ text_embeds.T  # (N, N)

    ranks = []
    for i in range(similarity.size(0)):
        sorted_indices = torch.argsort(similarity[i], descending=True)
        rank = (sorted_indices == i).nonzero(as_tuple=True)[0].item()
        ranks.append(rank)

    ranks = np.array(ranks)

    return {
        "R@1": np.mean(ranks < 1),
        "R@5": np.mean(ranks < 5),
        "R@10": np.mean(ranks < 10)
    }


In [9]:
from sklearn.metrics import accuracy_score, f1_score

def classification_metrics(image_embeds, text_embeds, labels):
    image_embeds = F.normalize(image_embeds, dim=1)
    text_embeds = F.normalize(text_embeds, dim=1)

    similarity = image_embeds @ text_embeds.T
    preds = similarity.argmax(dim=1).numpy()

    y_true = np.arange(len(labels))
    y_pred = preds

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "F1 Macro": f1_score(y_true, y_pred, average="macro"),
        "F1 Weighted": f1_score(y_true, y_pred, average="weighted"),
    }


In [10]:
CSV_PATH = "/content/image_text.csv"

dataset = NavarasaDataset(CSV_PATH)
dataloader = DataLoader(dataset, batch_size=16, shuffle=False)

image_embeds, text_embeds, labels = encode_dataset(dataloader)

retrieval = retrieval_metrics(image_embeds, text_embeds)
classification = classification_metrics(image_embeds, text_embeds, labels)

print("Retrieval Metrics")
for k, v in retrieval.items():
    print(f"{k}: {v:.4f}")

print("\nClassification Metrics")
for k, v in classification.items():
    print(f"{k}: {v:.4f}")

  0%|          | 0/871 [00:00<?, ?it/s]


FileNotFoundError: [Errno 2] No such file or directory: 'Bhayanaka/fear (121).jpg'